# Solar Panel Placement PCA

Parallel cellular algorithm (PCA) for solar-panel placement on a synthetic irradiance grid. Each cell holds a candidate layout; at every generation, cells update synchronously from a Moore-neighborhood guide. Edit grid dimensions, panel count, shade zones, and iteration settings.

Run the code cell to generate **`solar-panel-placement-pca.png`** in the repository folder.

In [ ]:
# Editable inputs
ROWS,COLS,PANELS=14,18,8
SHADE_CENTERS=[(4,5),(10,13)]
CELL_ROWS,CELL_COLS,ITERATIONS,SEED=6,6,180,19
OUTPUT="solar-panel-placement-pca.png"
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)
if not 1<=PANELS<=ROWS*COLS: raise ValueError("Panels must fit in the grid")
r,c=np.mgrid[0:ROWS,0:COLS]; sun=.72+.28*r/max(1,ROWS-1)+.05*np.sin(c/2)
for sr,sc in SHADE_CENTERS:sun-=.65*np.exp(-((r-sr)**2+(c-sc)**2)/7)
sun=np.clip(sun,.05,1)
def state():return rng.choice(ROWS*COLS,PANELS,False)
def value(x):return sun[np.unravel_index(x,(ROWS,COLS))].sum()
cells=[[state() for _ in range(CELL_COLS)] for _ in range(CELL_ROWS)]; best=max((x for row in cells for x in row),key=value); hist=[value(best)]
for _ in range(ITERATIONS):
 old=[[x.copy() for x in row] for row in cells]; new=[[None]*CELL_COLS for _ in range(CELL_ROWS)]
 for i in range(CELL_ROWS):
  for j in range(CELL_COLS):
   hood=[old[x][y] for x in range(max(0,i-1),min(CELL_ROWS,i+2)) for y in range(max(0,j-1),min(CELL_COLS,j+2)) if (x,y)!=(i,j)]; guide=max(hood,key=value) if hood else old[i][j]; child=old[i][j].copy(); ids=rng.choice(PANELS,max(1,PANELS//4),False); available=np.setdiff1d(np.arange(ROWS*COLS),np.delete(child,ids)); child[ids]=rng.choice(available,len(ids),False)
   for k in rng.choice(PANELS,min(2,PANELS),False):
    if rng.random()<.5 and guide[k] not in np.delete(child,k):child[k]=guide[k]
   new[i][j]=child if value(child)>=value(old[i][j]) else old[i][j]
 cells=new; candidate=max((x for row in cells for x in row),key=value)
 if value(candidate)>value(best):best=candidate.copy()
 hist.append(value(best))
pr,pc=np.unravel_index(best,(ROWS,COLS)); print(f"Panels: {PANELS}; normalized energy: {value(best):.2f}/{PANELS}")
fig,(a,b)=plt.subplots(1,2,figsize=(12,5)); im=a.imshow(sun,origin="lower",cmap="YlOrRd",vmin=0,vmax=1); a.scatter(pc,pr,marker="s",s=130,c="#1677a8",edgecolor="white"); a.set(title="Irradiance and panel locations",xlabel="Column",ylabel="Row"); fig.colorbar(im,ax=a,label="Relative irradiance"); b.plot(hist); b.set(title="Best layout energy",xlabel="Iteration",ylabel="Energy"); b.grid(alpha=.2); fig.tight_layout(); fig.savefig(OUTPUT,dpi=160); plt.show(); print("Saved",OUTPUT)